# Peak to gene (TSS +- 250kb for long-range CRE-peak capture & TF-Motif scan in age-specific open chromatin)

In [ ]:
!hostname

In [ ]:
import scanpy as sc
import scipy.sparse as sp
from pathlib import Path
import numpy as np
import pandas as pd
import anndata as ad
ad.settings.allow_write_nullable_strings = True
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
import muon as mu
# Import a module with ATAC-seq-related functions
from muon import atac as ac

In [ ]:
import os
os.chdir('/projects/bhdw/asachan/methods/FIREFate/moscot')
import sys
import logging
import warnings

objects_dir = "/work/hdd/bgdb/asachan/datasets_proj/SKM_ageing_human"
out_tmp = '/projects/bhdw/asachan/tmp'
figures_dir = '/work/hdd/bhdw/asachan/plot_outs/earthDT'

In [ ]:
pd.set_option('mode.string_storage', 'python')

#### TSS finder

In [ ]:
import numpy as np, pandas as pd, scipy.sparse as sp, pyranges as pr
from tqdm.auto import tqdm

# TSS lookup
gtf = pr.read_gtf(f"{human_genome_path}/gencode.v46.annotation.gtf").df
gtf = gtf[gtf.Feature == "gene"][["gene_id","Chromosome","Start","End","Strand"]]
gtf["TSS"]     = np.where(gtf.Strand == "+", gtf.Start, gtf.End)
gtf["gene_id"] = gtf["gene_id"].str.split(".").str[0]
tss = gtf.drop_duplicates("gene_id").set_index("gene_id")[["Chromosome","TSS"]]

atac, rna = mome["atac"], mome["rna"]
genes_e   = rna.var["ensg"].astype(str).values
peak_mid  = ((atac.var["start"].astype(int) + atac.var["end"].astype(int)) // 2).values
peak_chr  = atac.var["chrom"].astype(str).values
peak_by_chr = {ch: (np.where(peak_chr==ch)[0], peak_mid[peak_chr==ch])
               for ch in np.unique(peak_chr)}

WIN, SCALE = 250_000, 50_000 # distance -decay based likelihood
present = np.isin(genes_e, tss.index.values)
print(f"{present.sum():,}/{len(genes_e):,} genes have TSS")

cand_p, cand_g, cand_w = [], [], []
for gi in tqdm(np.where(present)[0], desc="P2G distance-decay"):
    e = genes_e[gi]
    chrom, t = tss.at[e, "Chromosome"], int(tss.at[e, "TSS"])
    if chrom not in peak_by_chr: continue
    p_idx, p_mid = peak_by_chr[chrom]
    d = np.abs(p_mid - t)
    m = d <= WIN
    if m.any():
        cand_p.append(p_idx[m])
        cand_g.append(np.full(m.sum(), gi, dtype=np.int64))
        cand_w.append(np.exp(-d[m] / SCALE).astype(np.float32))

cand_p = np.concatenate(cand_p); cand_g = np.concatenate(cand_g); cand_w = np.concatenate(cand_w)
P2G = sp.csr_matrix((cand_w, (cand_p, cand_g)), shape=(atac.n_vars, rna.n_vars))
print("P2G:", P2G.shape, "nnz:", P2G.nnz)
sp.save_npz(f"{out_tmp}/P2G.npz", P2G)

In [ ]:
import numpy as np, scipy.sparse as sp, pandas as pd
import matplotlib.pyplot as plt

X_open = mome["atac"].X.toarray() if sp.issparse(mome["atac"].X) else mome["atac"].X
# (n_cells, n_peaks) @ (n_peaks, n_genes) = (n_cells, n_genes) regulatory mass
reg_mass = X_open @ P2G                                                      # dense

ages = mome["rna"].obs["age_categorical"].astype(str).values
mask80 = ages == "80"; mask34 = ages == "34"

mean80 = reg_mass[mask80].mean(0); mean34 = reg_mass[mask34].mean(0)
log2fc = np.log2((mean80 + 1e-6) / (mean34 + 1e-6))

# rough significance via Welch t-test
from scipy.stats import ttest_ind
_, pvals = ttest_ind(reg_mass[mask80], reg_mass[mask34], axis=0, equal_var=False)
nlogp = -np.log10(np.clip(pvals, 1e-50, 1))


In [ ]:
# Per-RNA-cell per-gene peak mass across donor samples - the signal that
# `build_attn_keep_mask` actually filters on (it gates each gene-token by
# reg_mass[c, g] >= sample_threshold). Two cell-level summaries that are NOT
# 1/n_rna artifacts:
#   (a) total peak mass per cell        = reg_mass.sum(axis=1)
#   (b) mean peak mass per nonzero gene = reg_mass.sum / nnz per cell
import matplotlib.pyplot as plt
import numpy as np, pandas as pd, scipy.sparse as sp

RM             = reg_mass if sp.issparse(reg_mass) else sp.csr_matrix(reg_mass)
total_per_cell = np.asarray(RM.sum(axis=1)).ravel()
nnz_per_cell   = np.asarray((RM > 0).sum(axis=1)).ravel()
mean_per_gene  = np.where(nnz_per_cell > 0,
                          total_per_cell / np.maximum(nnz_per_cell, 1),
                          0.0)

samples_ord = sorted(np.unique(rna_sample),
                     key=lambda s: ({"Y": 0, "M": 1, "O": 2}.get(s[0], 9), s))
palette = {"Y": "#5fa8d3", "M": "#9c89b8", "O": "#b5651d"}
colors  = [palette.get(s[0], "grey") for s in samples_ord]

df = pd.DataFrame({"sample": rna_sample,
                   "total": total_per_cell,
                   "mean_per_gene": mean_per_gene})

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
for ax, col, ylab in [
    (axes[0], "total",
     "total peak mass per RNA cell\n(reg_mass.sum across genes)"),
    (axes[1], "mean_per_gene",
     "mean peak mass per nonzero gene\n(reg_mass.sum / nnz per cell)"),
]:
    data = [df.loc[df["sample"] == s, col].values for s in samples_ord]
    parts = ax.violinplot(data, showmedians=True, showextrema=False, widths=0.85)
    for pc, c in zip(parts["bodies"], colors):
        pc.set_facecolor(c); pc.set_edgecolor(c); pc.set_alpha(0.65)
    parts["cmedians"].set_color("k")
    for i, s in enumerate(samples_ord, start=1):
        sub = df[col][df["sample"] == s]
        ax.scatter([i], [sub.mean()], color="k", marker="D", s=20, zorder=5)
        ax.annotate(f"n={len(sub)}", (i, sub.max()),
                    ha="center", va="bottom", fontsize=9, color="dimgray")
    ax.set_xticks(range(1, len(samples_ord) + 1))
    ax.set_xticklabels(samples_ord)
    ax.set_ylabel(ylab)
    ax.set_xlabel("donor sample")

fig.suptitle("Per-RNA-cell per-gene peak mass",
             y=1.02)
plt.tight_layout()
# save in the figure directory
fig.savefig(f"{figures_dir}/per_rna_cell_per_gene_peak_mass.svg", format="svg", bbox_inches="tight")
plt.show()

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from matplotlib.offsetbox import AnchoredText

hits = (np.abs(log2fc) > 0.5) & (pvals < 1e-5)

N = 5
nlogp_safe = np.where(hits, nlogp, -np.inf)
top_up   = np.argsort(np.where(log2fc > 0, nlogp_safe, -np.inf))[::-1][:N]
top_down = np.argsort(np.where(log2fc < 0, nlogp_safe, -np.inf))[::-1][:N]

top_df = pd.DataFrame({
    "gene":      np.concatenate([mome["rna"].var_names.values[top_up],
                                 mome["rna"].var_names.values[top_down]]),
    "log2fc":    np.concatenate([log2fc[top_up],   log2fc[top_down]]),
    "-log10(p)": np.concatenate([nlogp[top_up],    nlogp[top_down]]),
    "side":      ["80↑"] * N + ["34↑"] * N,
})
print(top_df)

fig, ax = plt.subplots(figsize=(8, 5.5))
ax.scatter(log2fc, nlogp, s=4, alpha=0.4, c="grey")
ax.scatter(log2fc[hits], nlogp[hits], s=8, c="crimson")
ax.axhline(-np.log10(1e-5), ls="--", c="k", lw=0.6)
ax.axvline( 0.5, ls="--", c="k", lw=0.6); ax.axvline(-0.5, ls="--", c="k", lw=0.6)
ax.set_xlabel("log2( mass[80] / mass[34] )")
ax.set_ylabel("-log10(p)")
ax.set_title("Differential per-gene peak mass transported through GW-OT: 80yo vs 34yo")

# legend-style text boxes inside the plot
def gene_box(genes, color, header, loc):
    text = header + "\n" + "\n".join(genes)
    box  = AnchoredText(
        text, loc=loc, prop=dict(color=color, fontsize=10, fontweight="bold"),
        frameon=True, borderpad=0.5, pad=0.4,
    )
    box.patch.set_boxstyle("round,pad=0.3")
    box.patch.set_edgecolor(color)
    box.patch.set_facecolor("white")
    box.patch.set_alpha(0.85)
    return box

ax.add_artist(gene_box(
    [mome["rna"].var_names[gi] for gi in top_down],
    color="darkred",  header="Downregulated open-TSS sites in old", loc="upper left"
))
ax.add_artist(gene_box(
    [mome["rna"].var_names[gi] for gi in top_up],
    color="darkblue", header="Upregulated open-TSS sites in old",  loc="upper right"
))

plt.tight_layout()
fig.savefig(f"{figures_dir}/volcano_p2g_age.svg", format="svg", bbox_inches="tight")
plt.show()

In [ ]:
# Save the prep bundle that maxtoki_multimodal_prep.ipynb consumes. Single
# contract: rna_sub.h5ad (vocab-subset RNA, sorted by token id), mome.h5mu
# (rna + atac with X_peaks_rna), the three sparse arrays, and a manifest.
import json, scipy.sparse as sp
from pathlib import Path

PREP_DIR = Path(out_tmp) / "atac_rna_pairing_skm_prep"
PREP_DIR.mkdir(parents=True, exist_ok=True)

# 1. AnnData inputs to MaxToki RVE tokenization
# Verify the layer that downstream RVE depends on -- fail loud, never silent.
assert "counts" in rna_sub.layers, \
    "rna_sub.layers['counts'] missing -- RVE tokenization in the maxtoki notebook will mistokenize."
print(f"[save] rna_sub layers: {list(rna_sub.layers.keys())}")
rna_sub.write_h5ad(PREP_DIR / "rna_sub.h5ad")          # vocab-subset RNA + layers['counts']
mome.write_h5mu   (PREP_DIR / "mome.h5mu")             # MuData (rna + atac with X_peaks_rna)

# 2. Sparse arrays the sparsity-bias builder needs
sp.save_npz(PREP_DIR / "P2G.npz",         P2G.tocsr())
sp.save_npz(PREP_DIR / "X_peaks_rna.npz", sp.csr_matrix(X_peaks_rna))
sp.save_npz(PREP_DIR / "reg_mass.npz",    sp.csr_matrix(reg_mass))

# 3. Manifest — recipe + provenance
prep_manifest = {
    "format_version": 1,
    "n_cells": int(rna_sub.n_obs),
    "n_genes": int(rna_sub.n_vars),
    "n_peaks": int(mome["atac"].n_vars),
    "ot_pairing": {
        "stratification": "sample (per-donor GW couplings, sample-block-diagonal)",
        "T_dir":          str(out_tmp),
    },
    "p2g": {
        "win_bp":   250_000,
        "scale_bp": 50_000,
        "decay":    "exp(-d/scale)",
        "shape":    list(P2G.shape),
        "nnz":      int(P2G.nnz),
    },
    "files": {
        "rna_sub":     "rna_sub.h5ad     -- vocab-subset RNA AnnData, sorted by maxtoki_token; layers['counts'] = raw integer counts (REQUIRED by maxtoki RVE); obs.age_categorical, obs.sample; var.ensg, var.maxtoki_token, var.ensembl_id",
        "mome":        "mome.h5mu        -- MuData (rna + atac with X_peaks_rna; tfidf in atac.layers)",
        "P2G":         "P2G.npz          -- csr (n_peaks, n_genes_rna_sub) distance-decay weights",
        "X_peaks_rna": "X_peaks_rna.npz  -- csr (n_cells, n_peaks) per-RNA-cell soft peak accessibility",
        "reg_mass":    "reg_mass.npz     -- csr (n_cells, n_genes_rna_sub) per-gene peak mass per cell (= X_peaks_rna @ P2G)",
    },
}
(PREP_DIR / "manifest.json").write_text(json.dumps(prep_manifest, indent=2))
print(f"prep bundle: {PREP_DIR}")
for p in sorted(PREP_DIR.iterdir()):
    print(f"  {p.name}  ({p.stat().st_size/1e6:.1f} MB)")